In [ ]:
import numpy as np  #Importing required libraries and mnist dataset
import cv2
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt
import os


(imgtrain, labeltrain), (imgtest, labeltest) = mnist.load_data() #Loading training and testing labels and images


print("Input MNIST training data shape:", imgtrain.shape)  #Printing this message for verification that dataset is correct
print("Input MNIST testing data shape:", imgtest.shape)


def getforegroundmasks(images): #Defining a function for ostu's thresholding
    masks = [] #making an empty array for storing mask
    for img in images:
        img = img.astype(np.uint8) #ensuring that each pixel is 8 bit


        threshold, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU) #applying ostu's thresholding




        masks.append(mask) #putting all masks in array created earlier
    return np.array(masks)


trainmasks = getforegroundmasks(imgtrain) #getting masks for training images
testmasks = getforegroundmasks(imgtest) #getting masks for testing images


print("Foreground mask training data shape:", trainmasks.shape) #Printing size of masks created which wil be similar to input dataset
print("Foreground mask testing data shape:", testmasks.shape)


def generatecirclemasks(masks): #Defining function to generate tight enclosing circle
    circlemasks = [] #creating an array to store circular masks
    for mask in masks:
        mask = mask.astype(np.uint8) #ensuring that each mask is of 8 bits
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) #finding boarders of image


        if len(contours) > 0: #If such contours found then
            actcontour = max(contours, key=cv2.contourArea) #getting actual contour of digit from maximum of all contours
            (xcent, ycent), radius = cv2.minEnclosingCircle(actcontour) #computing parameters of minimum enclosing circle of actcontour
            center = (int(xcent), int(ycent)) #getting int value from float
            radius = int(radius) #getting int value of radius
        else:
            center, radius = (14, 14), 0 #In case no contour found then use these parameters


        circlemask = np.zeros_like(mask) #Creating blank mask for circle
        cv2.circle(circlemask, center, radius, 255, -1)  #Making white circle in the mask


        '''
        #enhancing circular masks with morphological operations
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)) #Stucturing element
        circlemask = cv2.dilate(circlemask, kernel, iterations=1) #Using dilation function to smooth circle contour
        '''
        circlemasks.append(circlemask) #Storing mask in the array of circlemasks


    return np.array(circlemasks) #returing array


traincirclemasks = generatecirclemasks(trainmasks) #generating circular masks for training masks
testcirclemasks = generatecirclemasks(testmasks) #generating circular masks for testing masks


print("Train circle masks shape:", traincirclemasks.shape) #Getting size of circular masks which will similar to input masks


n = 5 #Some example results
plt.figure(figsize=(10, 6))
for i in range(n):
    plt.subplot(3, n, i+1)
    plt.imshow(imgtrain[i], cmap='gray')
    plt.title(f"Input MNIST ({labeltrain[i]})") #Plotting input mnist images with thier classes
    plt.axis('off')


    plt.subplot(3, n, i+1+n)
    plt.imshow(trainmasks[i], cmap='gray')
    plt.title("Foreground Mask")#Plttoing foreground masks created
    plt.axis('off')


    plt.subplot(3, n, i+1+2*n)
    plt.imshow(traincirclemasks[i], cmap='gray')
    plt.title("Circle Mask") #Plotting corresponding circular masks created
    plt.axis('off')


plt.tight_layout()
plt.show() #Displaying results


os.makedirs("mnistcirclizeddataset", exist_ok=True)
np.save("mnistcirclizeddataset/trainimages.npy", imgtrain)
np.save("mnistcirclizeddataset/traincirclemasks.npy", traincirclemasks)
np.save("mnistcirclizeddataset/trainlabels.npy", labeltrain)


np.save("mnistcirclizeddataset/testimages.npy", imgtest)
np.save("mnistcirclizeddataset/testcirclemasks.npy", testcirclemasks)
np.save("mnistcirclizeddataset/testlabels.npy", labeltest)


print("Unique labels in training data:", np.unique(labeltrain)) #Printing classes present
